[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/astronerd21/Deep-Learning-Oil-Slick-Detection/blob/main/Lookalike_Analysis.ipynb)

# False Positive / Lookalike Analysis
**Goal:** Inspect "NO OIL" (0) samples that the model incorrectly predicted as "OIL" (1).
We will load the top misclassified samples exported by our evaluation script, apply Z-Score normalization, and visualize their radar signatures (VV and VH channels) alongside their pixel intensity histograms to categorize lookalikes (e.g., natural slicks, low-wind areas, ship wakes).

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

!pip install rasterio

In [ ]:
import os

# 1. Create the target directory for the extracted images
os.makedirs('/content/data', exist_ok=True)

# 2. Extract satellite images from Google Drive (if not already present)
if not os.path.exists('/content/data/images_s1'):
    print("Extracting satellite images from Google Drive (this will take a moment)...")
    !tar -xf /content/drive/MyDrive/OilSlickProject/data/OilSlick/OilSlick-images_s1-00.tar -C /content/data/
    !tar -xf /content/drive/MyDrive/OilSlickProject/data/OilSlick/OilSlick-images_s1-01.tar -C /content/data/
else:
    print("Images are already extracted!")

# 3. Create the target directory for the evaluation text file
os.makedirs('/content/evaluation_outputs', exist_ok=True)
print("\n✅ Environment setup complete!")
print("IMPORTANT: Please upload the 'false_positives_val_clean.txt' file into the 'evaluation_outputs' folder via the file menu on the left!")

In [ ]:
import os
from pathlib import Path
import rasterio
import numpy as np
import matplotlib.pyplot as plt

# Configuration and Paths
data_dir = Path("/content/data/images_s1/")
fp_list_path = Path("/content/evaluation_outputs/false_positives_val_clean.txt")

# Read the False Positive IDs exported by src/evaluate.py
if not fp_list_path.is_file():
    raise FileNotFoundError(f"Missing false positive list at {fp_list_path}. Make sure you uploaded it!")

with open(fp_list_path, "r") as f:
    fp_filenames = [line.strip() for line in f if line.strip()]

print(f"Total False Positives found: {len(fp_filenames)}")

# Select the top 10 misclassifications for visual inspection
top_10_fps = fp_filenames[:10]
print(f"Inspecting the first {len(top_10_fps)} samples...")

In [ ]:
def load_and_normalize_sar(filepath: Path) -> np.ndarray:
    """
    Loads a 2-band GeoTIFF and applies channel-wise Z-score normalization.
    This exactly mirrors the _load_geotiff logic in src/dataset.py.
    """
    with rasterio.open(filepath) as src:
        data = src.read().astype(np.float32)  # Shape: (2, H, W)
        
    # Apply Z-score normalization per channel (VV=0, VH=1)
    for c in range(data.shape[0]):
        channel_mean = np.mean(data[c])
        channel_std = np.std(data[c])
        if channel_std > 0:
            data[c] = (data[c] - channel_mean) / channel_std
            
    return data

In [ ]:
# Plotting Grid: False Positives alongside their histograms
# We will create a grid where each row represents one False Positive sample.
# Columns: [VV Image] | [VV Histogram] | [VH Image] | [VH Histogram]

num_samples = len(top_10_fps)
fig, axes = plt.subplots(num_samples, 4, figsize=(20, 4 * num_samples))
fig.suptitle("Lookalike Analysis: Normalized Radar Signatures & Histograms", fontsize=16, y=1.02)

for idx, filename in enumerate(top_10_fps):
    filepath = data_dir / filename
    
    if not filepath.exists():
        print(f"File not found: {filepath}")
        continue
        
    data = load_and_normalize_sar(filepath)
    vv_band = data[0]
    vh_band = data[1]
    
    # 1) VV Channel Image
    ax_vv = axes[idx, 0]
    im_vv = ax_vv.imshow(vv_band, cmap='gray')
    ax_vv.set_title(f"VV Channel | {filename}")
    ax_vv.axis('off')
    
    # 2) VV Channel Histogram
    ax_vv_hist = axes[idx, 1]
    ax_vv_hist.hist(vv_band.ravel(), bins=50, color='blue', alpha=0.7)
    ax_vv_hist.set_title("VV Histogram (Z-Scored)")
    ax_vv_hist.grid(True, alpha=0.3)
    
    # 3) VH Channel Image
    ax_vh = axes[idx, 2]
    im_vh = ax_vh.imshow(vh_band, cmap='gray')
    ax_vh.set_title(f"VH Channel | {filename}")
    ax_vh.axis('off')
    
    # 4) VH Channel Histogram
    ax_vh_hist = axes[idx, 3]
    ax_vh_hist.hist(vh_band.ravel(), bins=50, color='orange', alpha=0.7)
    ax_vh_hist.set_title("VH Histogram (Z-Scored)")
    ax_vh_hist.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

### 📝 Visual Categorization Results

Based on the visual signatures and histograms above, we categorize the top 10 misclassifications into the following maritime phenomena:

1. **Low-Wind Calm Areas (Lookalikes)**
   * *Characteristics:* Large, irregular dark patches in the VV channel (due to lack of capillary waves/Bragg scattering) but often indistinguishable from noise in the VH channel.
   * *Sample IDs:* `[Insert filenames here]`

2. **Ship Wakes**
   * *Characteristics:* Linear V-shaped dark or bright features, often trailing a bright point target (the vessel itself).
   * *Sample IDs:* `[Insert filenames here]`

3. **Natural Biogenic Slicks**
   * *Characteristics:* Thin, winding, filament-like dark structures caused by algal blooms or fish oils. Highly similar to anthropogenic oil spills but often more feathery at the edges.
   * *Sample IDs:* `[Insert filenames here]`
   
4. **Internal Waves / Bathymetric Features**
   * *Characteristics:* Periodic banding of alternating dark and light stripes.
   * *Sample IDs:* `[Insert filenames here]`